# Outputs driving a Python loop

This example wires a widget output back into a Python environment step. It needs packages that
Vibe Widget does not depend on, so install them first:

```
pip install "numpy<2" "gym==0.26.2" "gym-super-mario-bros==7.4.0" "nes-py==8.2.1"
```

The pattern itself is general: declare an input per value the widget should show, declare an output
for the user's choice, observe that output, and assign the input traits to push new values back.


In [ ]:
pip install "numpy<2" "gym==0.26.2" "gym-super-mario-bros==7.4.0" "nes-py==8.2.1"

In [ ]:
import vibe_widget as vw
import os
import dotenv
dotenv.load_dotenv()
vw.config(api_key=os.getenv("OPENROUTER_API_KEY"))

In [ ]:
  import gym
  import gym_super_mario_bros
  from nes_py.wrappers import JoypadSpace
  from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
  import vibe_widget as vw
  from vibe_widget.utils.serialization import clean_for_json

  env = JoypadSpace(gym_super_mario_bros.make("SuperMarioBros-1-1-v0"), SIMPLE_MOVEMENT)

  reset_out = env.reset()
  if isinstance(reset_out, tuple):
      obs = reset_out[0]
      info = reset_out[1] if len(reset_out) > 1 else {}
  else:
      obs, info = reset_out, {}

  widget = vw.create(
      """Create a Mario viewer:
      - Show the current frame from 'frame'
      - Show score, coins, and time from 'info'
      - Provide action buttons labeled from 'action_names'
      - Output the selected action index as 'action'""",
     inputs=vw.inputs(
            frame=obs,
            info=info,
            action_names=[str(a) for a in SIMPLE_MOVEMENT],
      ),
      cache=False,
      outputs=vw.outputs(action="index of selected action"),
  )

  def step_env(event):
      step_out = env.step(event.new)
      if len(step_out) == 5:
          obs, reward, terminated, truncated, info = step_out
          done = terminated or truncated
      else:
          obs, reward, done, info = step_out

      widget.frame = clean_for_json(obs)
      widget.info = clean_for_json(info)

      if done:
          reset_out = env.reset()
          if isinstance(reset_out, tuple):
              obs = reset_out[0]
              info = reset_out[1] if len(reset_out) > 1 else {}
          else:
              obs, info = reset_out, {}
          widget.frame = clean_for_json(obs)
          widget.info = clean_for_json(info)

  widget.outputs.action.observe(step_env)

In [ ]:
  print(widget.status)
  print(widget.error_message)
  print(widget.logs[-5:])